# LocalSparse M1.5 — Unified Veyra3 benchmark + Gemma 4 E2B validation

Single-notebook pipeline (plan §7.6). Run cells top-to-bottom; each one prints `=== [STAGE X] ===` so you can spot where it stopped if interrupted.

**Outline:**
1. Setup (clone, install, HF token, GPU check)
2. Config — pick MODEL and PHASE
3. Phase A — Veyra3 deep benchmark (`bench_veyra3.py --section all`)
4. Phase B — Gemma 4 local smoke (synthetic config, no download)
5. Phase C — Gemma 4 G6 validation on real weights (gated)
6. Final summary

**Decision gates** (from plan §7.8): if G-A2 fails, stop; investigate KV-injection before spending Gemma 4 budget. If C3 smoke fails, stop; surgery is wrong.

## 1. Setup

In [ ]:
# Clone + install
!rm -rf /content/localsparse 2>/dev/null
!git clone https://github.com/kaaninel/localsparse.git /content/localsparse
%cd /content/localsparse
!pip install -q -e . accelerate 'torch>=2.4'
!pip install -q --upgrade 'git+https://github.com/huggingface/transformers.git'
import torch
print('=== [STAGE SETUP] ===')
print(f'torch: {torch.__version__}')
print(f'cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu:  {torch.cuda.get_device_name(0)}')

In [ ]:
# HF token (needed for gated Gemma 4)
import os
from getpass import getpass
if 'HF_TOKEN' not in os.environ:
    try:
        os.environ['HF_TOKEN'] = getpass('HF token (leave blank to skip Phase C): ')
    except Exception:
        os.environ['HF_TOKEN'] = ''
print('HF token set:', bool(os.environ.get('HF_TOKEN')))

## 2. Config

In [ ]:
# Pick which phases to run.
# RUN_A = True   # Veyra3 deep benchmark (always cheap)
# RUN_B = True   # Gemma 4 local smoke (synthetic, instant)
# RUN_C = True   # Gemma 4 real G6 (needs HF token + A100 for full budget)
RUN_A = True
RUN_B = True
RUN_C = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'bfloat16' if torch.cuda.is_available() else 'float32'
GEMMA_MODEL_ID = 'google/gemma-4-E2B'   # change to '-it' for instruction-tuned
print(f'config: device={DEVICE} dtype={DTYPE} model={GEMMA_MODEL_ID}')

## 3. Phase A — Veyra3 deep benchmark

Runs A0..A8 with real budgets. A0 establishes the convergence baseline; A2 is the central go/no-go gate. **Expect ~30–60 min on A100, longer on T4.**

In [ ]:
if RUN_A:
    print('=== [STAGE PHASE A] ===')
    !python scripts/bench_veyra3.py --section all --device {DEVICE} --dtype {DTYPE} --run_dir /content/runs/m15/phase_a
    # show REPORT.md
    import glob, os
    reports = sorted(glob.glob('/content/runs/m15/phase_a/REPORT.md'))
    if reports:
        print('\n--- REPORT.md ---')
        print(open(reports[0]).read())
else:
    print('skipping Phase A')

In [ ]:
# Decision gate: G-A2 (central). Show kv/weights ratio.
import json, pathlib
if RUN_A:
    a2_path = pathlib.Path('/content/runs/m15/phase_a/a2.json')
    if a2_path.exists():
        a2 = json.loads(a2_path.read_text())
        ratio = a2.get('ratios', {}).get('kv_over_weights', 0.0)
        print(f'G-A2 ratio (kv/weights): {ratio:.3f}')
        print(f'G-A2 verdict: {"PASS" if ratio >= 0.5 else "FAIL — investigate before Phase C"}')

## 4. Phase B — Gemma 4 local smoke (synthetic, instant)

In [ ]:
if RUN_B:
    print('=== [STAGE PHASE B] ===')
    !python scripts/gemma4_local_smoke.py
else:
    print('skipping Phase B')

## 5. Phase C — Gemma 4 E2B validation

Runs C3 smoke → C1 prebench → C2 real G6.  
C3 stops the chain if it fails (catches surgery bugs early).

In [ ]:
if RUN_C:
    print('=== [STAGE PHASE C] ===')
    !python scripts/bench_gemma4.py \
        --section all \
        --model_id {GEMMA_MODEL_ID} \
        --device {DEVICE} --dtype {DTYPE} \
        --run_dir /content/runs/m15/phase_c \
        --c2_n_facts 512 \
        --c2_max_steps 4000 \
        --c2_batch_size 4 \
        --c2_seq_len 512 \
        --c2_bank_max_length 1024 \
        --c2_threshold 0.6
else:
    print('skipping Phase C')

## 6. Final summary

In [ ]:
import json, pathlib
print('=== [STAGE FINAL SUMMARY] ===')
for label, path in [('Phase A', '/content/runs/m15/phase_a/summary.json'),
                    ('Phase C', '/content/runs/m15/phase_c/summary.json')]:
    p = pathlib.Path(path)
    if p.exists():
        data = json.loads(p.read_text())
        print(f'\n{label}:')
        for s, info in data.get('sections', {}).items():
            print(f'  {s}: {info.get("status")}')
    else:
        print(f'\n{label}: (not run)')

# Central verdict
a2 = pathlib.Path('/content/runs/m15/phase_a/a2.json')
c2 = pathlib.Path('/content/runs/m15/phase_c/c2.json')
print('\n--- CENTRAL VERDICTS ---')
if a2.exists():
    r = json.loads(a2.read_text()).get('ratios', {}).get('kv_over_weights', 0.0)
    print(f'G-A2 (Veyra3 5M, kv/weights):  {r:.3f}  [pass>=0.5]')
if c2.exists():
    c2d = json.loads(c2.read_text())
    print(f'G-C2 (Gemma 4 E2B, ratio):     {c2d.get("ratio", 0):.3f}  [pass>=0.6]')
    print(f'  weights_accuracy:            {c2d.get("weights_accuracy", 0):.3f}')
    print(f'  mount_accuracy:              {c2d.get("mount_accuracy", 0):.3f}')
    print(f'  control_accuracy:            {c2d.get("control_accuracy", 0):.3f}')
    print(f'  verdict:                     {c2d.get("status", "?").upper()}')